<a href="https://colab.research.google.com/github/look4pritam/NetflixPrize/blob/master/Notebooks/v1.0.0/v1.0.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Prize

## Install Python packages.

In [ ]:
!pip install numpy pandas scipy tqdm

In [ ]:
!pip install implicit

## Set the project root directory.

In [ ]:
import os

root_dir = '/content/'
os.chdir(root_dir)

!ls -al

## Download the Netflix Prize dataset from Kaggle.

### Set the Kaggle API key.

### Generate and upload 'kaggle.json' file.

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = root_dir
!chmod 600 /content/kaggle.json

### Download the Netflix Prize dataset.

In [ ]:
!kaggle datasets download netflix-inc/netflix-prize-data
!ls -al

In [ ]:
import os
import zipfile

### Extract the Netflix Prize dataset.

In [ ]:
zip_path = "netflix-prize-data.zip"
dataset_path = "netflix_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

print("Extracted Netflix Prize dataset from:", os.listdir(dataset_path))

## Load the Netflix Prize dataset.

### Import Python packages.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
def build_sparse_matrix(input_folder):

    user_ids = []
    movie_ids = []
    ratings = []

    user_map = {}
    movie_map = {}

    user_counter = 0
    movie_counter = 0

    files = [
        "combined_data_1.txt",
        "combined_data_2.txt",
        "combined_data_3.txt",
        "combined_data_4.txt"
    ]

    for file in files:

        path = os.path.join(input_folder, file)

        print("\nProcessing input file: ", file)

        movie_id = None

        with open(path, "r", encoding="latin-1") as f:

            for line in tqdm(f):

                line = line.strip()

                if not line:
                    continue

                # movie ID line
                if line.endswith(":"):

                    movie = int(line[:-1])

                    if movie not in movie_map:
                        movie_map[movie] = movie_counter
                        movie_counter += 1

                    movie_id = movie_map[movie]

                else:

                    parts = line.split(",")

                    if len(parts) < 2:
                        continue

                    user = int(parts[0])
                    rating = float(parts[1])

                    if user not in user_map:
                        user_map[user] = user_counter
                        user_counter += 1

                    user_ids.append(user_map[user])
                    movie_ids.append(movie_id)
                    ratings.append(rating)

    rating_matrix = coo_matrix(
        (ratings, (user_ids, movie_ids)),
        shape=(user_counter, movie_counter)
    )

    return rating_matrix.tocsr(), user_map, movie_map

In [ ]:
rating_matrix, user_map, movie_map = build_sparse_matrix("netflix_dataset")

print("Matrix shape: ", rating_matrix.shape)
print("Number of ratings: ", rating_matrix.nnz)

## Factorize matrix using Alternating Least Squares (ALS).

### Define alpha value.

In [ ]:
alpha = 40

In [ ]:
R_conf = rating_matrix * alpha

### Import 'AlternatingLeastSquares' algorithm.

In [ ]:
from implicit.als import AlternatingLeastSquares

In [ ]:
model = AlternatingLeastSquares(
    factors=100,
    regularization=0.1,
    iterations=20,
    use_gpu=True
)

model.fit(R_conf)

## Computer the RMSE metric.

In [ ]:
rows, cols = rating_matrix.nonzero()
ratings = rating_matrix.data

In [ ]:
user_factors = model.user_factors.to_numpy()
item_factors = model.item_factors.to_numpy()

In [ ]:
batch_size = 1_000_000
n = len(ratings)

sq_error = 0

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    r = rows[start:end]
    c = cols[start:end]

    preds = global_mean + np.einsum(
        "ij,ij->i",
        user_factors[r],
        item_factors[c]
    )

    preds = np.clip(preds, 1, 5)

    sq_error += np.sum((ratings[start:end] - preds) ** 2)



In [ ]:
rmse = np.sqrt(sq_error / n)
print("RMSE:", rmse)